# 03 · Baseline B1 – Whisper Translate (end-to-end)

**B1**: audio tiếng Việt → `openai/whisper-small` (`task="translate"`) → văn bản tiếng Anh, **không** qua transcript tiếng Việt và khối dịch.
So với **B0** (cascade): PhoWhisper-small → VinAI Translate gốc. Cùng 857 bản thu FLEURS vi_vn test, cùng sacreBLEU 2.4.3.
Proposal Task 1 đã nêu "Whisper task translate làm đối chứng" → B1 là baseline đối chứng theo proposal (không phải tái lập số của paper).

Notebook: (1) kiểm gói bàn giao `DANG_BASELINE_HANDOFF`, (2) chạy thử 5 mẫu, (3) chạy đủ 857, (4) chấm BLEU/chrF++, RTF, theo miền,
(5) bootstrap so với B0. Code B1: `scripts/b1/b1_whisper_translate.py` (dựa trên code của gói bàn giao, có sửa – xem mục 1).

In [1]:
import os, sys, json, csv, subprocess, time, shutil, statistics, re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Thư mục thực nghiệm (có scripts/, data/, checkpoints/): task3/ – notebook nằm ở task3/notebooks/
T3 = next(c for c in [Path.cwd().parent, Path.cwd(), Path(r"D:\CAIDCM\teamwork\task3")] if (c / "scripts" / "mt" / "train_lora.py").exists())
os.chdir(T3); TW = T3.parent
PY = sys.executable
ENV = dict(os.environ, HF_HUB_CACHE=str(T3 / "hf_home" / "hub"), HF_HUB_OFFLINE="1", HF_HUB_DISABLE_TELEMETRY="1",
           PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
MTREV = "ae7baa85da07dbe8e23ac26a9f5ef560c17e2138"
BO_QUA = ("Warning", "warnings.warn", "warn(", "forced_decoder_ids", "attention mask", "fget.__get__", "TypedStorage")

def chay(*lenh, log=None):
    """Chạy 1 script của project (cùng venv), in log từng dòng, ghi thêm vào file log nếu có; lỗi thì dừng ô."""
    t0 = time.time()
    p = subprocess.Popen([PY, "-X", "utf8", *map(str, lenh)], cwd=T3, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace")
    fl = open(T3 / log, "a", encoding="utf-8") if log else None
    for d in p.stdout:
        if fl: fl.write(d)
        if not any(x in d for x in BO_QUA): print(d, end="")
    p.wait(); fl and fl.close()
    print(f"--- mã thoát {p.returncode} · {time.time() - t0:.0f} s")
    if p.returncode: raise RuntimeError(f"lệnh lỗi: {lenh}")

J = lambda p: json.loads(Path(p).read_text(encoding="utf-8"))
def mt(v, s):
    p = T3 / "runs/mt" / v / s / "score.json"
    return J(p) if p.exists() else None
vn = lambda x, d=2: "–" if x is None else f"{float(x):.{d}f}".replace(".", ",")
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})
import torch
print("Thư mục:", T3, "| Python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG")

Thư mục: D:\CAIDCM\teamwork\task3 | Python 3.11.1 | torch 2.0.1+cu117 | GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 1. Kiểm gói bàn giao baseline

| Hạng mục checklist | Gói bàn giao | Xử lý ở notebook này |
|---|---|---|
| Dataset 857 mẫu, không trùng | Có (`check_data.py`) | Kiểm lại ô dưới |
| B0 cascade | Có số 22,69 / 49,38 (`b0_summary.json`) | Đối chiếu file kết quả ô dưới |
| B1 chạy thật với audio | **Chưa chạy** (không có GPU) | Chạy ở mục 3 |
| Output theo schema chung | Chưa (tên trường khác) | Script B1 xuất đúng schema |
| Audio > 30 s | Code gốc `processor(audio)` **cắt còn 30 s** | Sửa: long-form như B0 |
| Đọc file trên Windows | `open()` không `encoding="utf-8"`; cần `torchaudio` (venv không có) | Sửa: utf-8 + soundfile |
| Revision, seed, suppress tokens | Không ghim | Ghim `973afd24`, seed 42, suppress tokens của protocol |
| Gap analysis | Ghi "chưa có BLEU dịch sạch" – **đã có** (B0 38,27) | Số đúng ở notebook 05 |

In [2]:
G = TW / "kiem_dang_baseline/goc/DANG_BASELINE_HANDOFF"
MAN = TW / "GUI_DANG_FLEURS_TEST/manifest_fleurs_test.jsonl"
m_dang = [json.loads(l) for l in open(MAN, encoding="utf-8")]
m_t3 = {json.loads(l)["id"].removesuffix(".wav"): json.loads(l) for l in open(T3 / "runs/manifests/asr_fleurs_test.jsonl", encoding="utf-8")}
print("Manifest bàn giao:", len(m_dang), "mẫu | trùng sample_id:", len(m_dang) - len({r["sample_id"] for r in m_dang}),
      "| khớp manifest thực nghiệm (id + câu Việt):", sum(r["sample_id"] in m_t3 and m_t3[r["sample_id"]]["reference"] == r["reference_vi"] for r in m_dang))
b0s = J(G / "b0_summary.json") if (G / "b0_summary.json").exists() else None
b0 = mt("B0", "fleurs_test_asr_pho")
print("B0 trong gói bàn giao:", b0s and b0s["metrics"], "| file kết quả:", {"BLEU": b0["BLEU"], "chrF++": b0["chrF++"], "n": b0["n"]})
print("→ khớp" if b0s and b0s["metrics"]["BLEU"] == b0["BLEU"] and b0s["metrics"]["chrF++"] == b0["chrF++"] else "→ KHÁC / không có gói")
print("B0 dịch transcript sạch (gói bàn giao ghi 'chưa có'):", mt("B0", "fleurs_test_clean")["BLEU"])

Manifest bàn giao: 857 mẫu | trùng sample_id: 0 | khớp manifest thực nghiệm (id + câu Việt): 857
B0 trong gói bàn giao: {'BLEU': 22.69, 'chrF++': 49.38, 'n_samples': 857} | file kết quả: {'BLEU': 22.69, 'chrF++': 49.38, 'n': 857}
→ khớp
B0 dịch transcript sạch (gói bàn giao ghi 'chưa có'): 38.27


## 2. Chạy thử 5 mẫu

In [3]:
thu = T3 / "runs/b1/smoke5.jsonl"; thu.unlink(missing_ok=True)
chay("scripts/b1/b1_whisper_translate.py", "--manifest", MAN, "--out", thu, "--limit", 5)
display(pd.read_json(thu, lines=True, dtype={"sample_id": str, "sent_id": str})[["sample_id", "audio_duration_s", "latency_s", "status", "pred_en", "reference_en"]])

  B1 0/5


B1_XONG D:\CAIDCM\teamwork\task3\runs\b1\smoke5.jsonl


--- mã thoát 0 · 18 s


,sample_id,audio_duration_s,latency_s,status,pred_en,reference_en
0,7702082596424665893,9.42,1.082,ok,"The other side of the face is a normal phenomenon in many countries, and the 5th round of communication is...","On the other hand, icy and snowy conditions are normal in many countries, and traffic goes on mostly unint..."
1,5471256245985564162,9.30,1.133,ok,"On the other hand, it is a common phenomenon in many countries, and it is almost impossible to communicate...","On the other hand, icy and snowy conditions are normal in many countries, and traffic goes on mostly unint..."
2,15589888555401605921,8.64,0.956,ok,"Other faces are usually in many countries, and in the year, it is almost impossible to communicate with ea...","On the other hand, icy and snowy conditions are normal in many countries, and traffic goes on mostly unint..."
3,910330178977835580,7.20,0.668,ok,This is an important way to distinguish between action and action.,This is an important way to distinguish between some verbs and objects.
4,9278960538865753016,5.88,0.648,ok,This is a very important way to distinguish between action and growth.,This is an important way to distinguish between some verbs and objects.


## 3. Chạy đủ 857 bản thu (GPU, ~10–20 phút; chạy lại sẽ bỏ qua mẫu đã xong)

In [4]:
PRED = T3 / "runs/b1/predictions_b1.jsonl"
chay("scripts/b1/b1_whisper_translate.py", "--manifest", MAN, "--out", PRED, log="logs/b1_whisper_translate.log")
b1 = pd.read_json(PRED, lines=True, dtype={"sample_id": str, "sent_id": str})
print("Số dòng:", len(b1), "| trạng thái:", b1.status.value_counts().to_dict(), "| long-form:", int(b1.long_form.sum()))
print(json.dumps(J(T3 / "runs/b1/run_metadata_b1.json"), ensure_ascii=False, indent=1)[:900])

B1_XONG D:\CAIDCM\teamwork\task3\runs\b1\predictions_b1.jsonl


--- mã thoát 0 · 7 s
Số dòng: 857 | trạng thái: {'ok': 857} | long-form: 7
{
 "model_id": "B1_whisper-small_translate@973afd24",
 "model": "openai/whisper-small",
 "revision": "973afd24965f72e36ca33b3055d56a652f456b4d",
 "generate": {
  "language": "vi",
  "task": "translate",
  "num_beams": 5,
  "do_sample": false,
  "max_new_tokens": 440
 },
 "dtype": "float16",
 "seed": 42,
 "suppress_sha256": "6f30bf8c441f5c66afa38f7eee6ad7af5433c3cd2ac76e1d5ad6e6aeac520f12",
 "manifest": "D:\\CAIDCM\\teamwork\\GUI_DANG_FLEURS_TEST\\manifest_fleurs_test.jsonl",
 "manifest_sha256": "7e885ae86c4963476b879d59e5771e929218596367f30513758cc62f6c95f300",
 "gpu": "NVIDIA GeForce RTX 4060 Laptop GPU",
 "torch": "2.0.1+cu117",
 "long_form": "> 30 s: truncation=False, padding=longest, return_timestamps=True (như B0)"
}


## 4. Chấm BLEU / chrF++ (sacreBLEU 2.4.3, cùng cấu hình B0) – lưu `runs/mt/B1_whisper-small_translate/fleurs_test_audio/`

In [5]:
import sacrebleu
assert len(b1) == 857 and (b1.status == "ok").all(), "chưa đủ 857 mẫu ok"
d = T3 / "runs/mt/B1_whisper-small_translate/fleurs_test_audio"; d.mkdir(parents=True, exist_ok=True)
with open(d / "hyp.jsonl", "w", encoding="utf-8") as f:   # cùng định dạng hyp.jsonl của B0/V* (id có đuôi .wav) để dùng chung bootstrap.py
    for r in b1.itertuples():
        f.write(json.dumps({"id": r.sample_id + ".wav", "sent_id": str(r.sent_id), "src": "", "ref": r.reference_en, "hyp": r.pred_en}, ensure_ascii=False) + "\n")
bl, ch = sacrebleu.BLEU(), sacrebleu.CHRF(word_order=2)
b, c = bl.corpus_score(list(b1.pred_en), [list(b1.reference_en)]), ch.corpus_score(list(b1.pred_en), [list(b1.reference_en)])
rtf = b1.latency_s.sum() / b1.audio_duration_s.sum()
sc = {"BLEU": round(b.score, 2), "chrF++": round(c.score, 2), "bleu_signature": str(bl.get_signature()), "chrf_signature": str(ch.get_signature()),
      "n": len(b1), "he": "B1 end-to-end Whisper-small translate", "model_id": b1.model_id.iloc[0], "RTF_tong": round(rtf, 3),
      "latency_p50_s": round(b1.latency_s.median(), 3), "latency_p95_s": round(b1.latency_s.quantile(.95), 3),
      "ghi_chu_RTF": "latency mỗi mẫu = chỉ bước generate (đồng bộ CUDA, đã khởi động GPU), batch 1; khác cách đo do_toc_do.py (100 mẫu, 3 lần xen kẽ)"}
(d / "score.json").write_text(json.dumps(sc, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: sc[k] for k in ("BLEU", "chrF++", "n", "RTF_tong", "latency_p50_s", "latency_p95_s")}, ensure_ascii=False))
nh = []
for ten, g in [("toàn bộ", b1)] + [(f"miền {k}", v) for k, v in b1.groupby("domain")] + [("≤ 30 s", b1[~b1.long_form]), ("> 30 s", b1[b1.long_form])]:
    nh.append({"nhóm": ten, "n": len(g), "BLEU": round(bl.corpus_score(list(g.pred_en), [list(g.reference_en)]).score, 2),
               "chrF++": round(ch.corpus_score(list(g.pred_en), [list(g.reference_en)]).score, 2)})
display(pd.DataFrame(nh))

{"BLEU": 3.8, "chrF++": 25.45, "n": 857, "RTF_tong": 0.137, "latency_p50_s": 0.734, "latency_p95_s": 8.196}


,nhóm,n,BLEU,chrF++
0,toàn bộ,857,3.80,25.45
1,miền du_lich,254,4.23,25.98
2,miền giao_duc,336,4.42,27.07
3,miền tin_tuc,267,2.86,23.17
4,≤ 30 s,850,3.85,25.52
5,> 30 s,7,2.31,22.91


## 5. So sánh với B0 – bootstrap 1000 lần theo cụm câu (`scripts/mt/bootstrap.py`)

In [6]:
B1H = "runs/mt/B1_whisper-small_translate/fleurs_test_audio/hyp.jsonl"
for ten, a in (("B0 cascade (PhoWhisper → VinAI gốc)", "runs/mt/B0/fleurs_test_asr_pho/hyp.jsonl"),
               ("Whisper-small → VinAI gốc (cascade, khối nghe thứ hai)", "runs/mt/B0/fleurs_test_asr_wsp/hyp.jsonl")):
    o = T3 / f"reports/mt/bootstrap_{Path(a).parent.parent.name}_{Path(a).parent.name}_vs_B1.json"
    if not o.exists():
        chay("scripts/mt/bootstrap.py", "--a", a, "--b", B1H, "--data", "runs/mt_data/fleurs_test_clean.jsonl", "--out", o, "--n", 1000)
    r = J(o); print(f"{ten}: {r['BLEU_A']} → B1 {r['BLEU_B']} | Δ {r['delta']:+.2f} [{r['ci95'][0]}; {r['ci95'][1]}] | p(Δ≤0) = {r['p_one_sided_delta_le_0']}")

B0 cascade (PhoWhisper → VinAI gốc): 22.69 → B1 3.8 | Δ -18.89 [-20.65; -17.14] | p(Δ≤0) = 1.0
Whisper-small → VinAI gốc (cascade, khối nghe thứ hai): 18.58 → B1 3.8 | Δ -14.78 [-16.11; -13.5] | p(Δ≤0) = 1.0


In [7]:
A = {json.loads(l)["id"]: json.loads(l) for l in open(T3 / "runs/mt/B0/fleurs_test_asr_pho/hyp.jsonl", encoding="utf-8")}
B = {json.loads(l)["id"]: json.loads(l) for l in open(T3 / B1H, encoding="utf-8")}
cf = sacrebleu.CHRF(word_order=2)
vd = pd.DataFrame([{"id": i, "tham_chieu": A[i]["ref"], "B0_cascade": A[i]["hyp"], "B1_end_to_end": B[i]["hyp"],
                    "chrF_B0": round(cf.sentence_score(A[i]["hyp"], [A[i]["ref"]]).score, 1), "chrF_B1": round(cf.sentence_score(B[i]["hyp"], [B[i]["ref"]]).score, 1)} for i in A])
print(f"B1 tốt hơn B0 ở {(vd.chrF_B1 > vd.chrF_B0).sum()} câu, kém hơn ở {(vd.chrF_B1 < vd.chrF_B0).sum()} câu (chrF++ từng câu).")
display(vd.assign(d=vd.chrF_B1 - vd.chrF_B0).sort_values("d").iloc[[0, 1, 2, -3, -2, -1]].drop(columns="d"))

B1 tốt hơn B0 ở 70 câu, kém hơn ở 787 câu (chrF++ từng câu).


,id,tham_chieu,B0_cascade,B1_end_to_end,chrF_B0,chrF_B1
326,12706544329768734968.wav,"As knowledge of Greek declined, the West found itself cut off from its Greek philosophical and scientific ...","As understanding of Greece declined, the West found itself cut off from its Greek philosophical and scient...","When I learned about Hela, I found out that I was able to learn Hela from the root of the root of the root...",84.8,3.9
329,15083026526454998988.wav,"As knowledge of Greek declined, the West found itself cut off from its Greek philosophical and scientific ...","As the understanding of Greece declined, the West found itself cut off from its Greek philosophical and sc...","When it comes to science and science, it is clear that the world has been cut off from the roots of scienc...",84.2,5.2
511,7500010259362029907.wav,"The two compounds react with one another to form crystals that may block kidney function, researchers at t...","The two compounds react with each other to form crystals that could impede kidney function, researchers at...",These two boxes must be attached to each other to form a body that can make a body that is capable of maki...,81.8,5.9
644,10751579734990864351.wav,"The vertical clearance under the bridge is 15 meters. Construction was completed in August 2011, it didn't...",The provincial height not below the bridge is m. The construction was completed in August of the year two ...,"The height of the sky below the bridge is 15 meters. The construction was completed in August 2011, but un...",42.7,58.1
180,12973626254752465669.wav,"Apia is the capital of Samoa. The town is on the island of Upolu and has a population of just under 40,000.",Aphia is the capital of salvo. This city is located on the island of ukraine and has a population of nearl...,Apia is the capital of Samoa. This city is located on the island of Upolu and has a population of nearly 4...,53.5,74.0
853,8180761900974663168.wav,Entertainment news website TMZ understands the photographer stopped his vehicle on the other side of Sepul...,A. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a. a...,"On the website, the name of the address, TMZ, is known, the photographer stopped the car on the other side...",1.2,44.8
